# Plant Disease Detection — Deep Learning Experiments

**Author:** Farrel Ghozy Affifudin (452024611053)
**Course:** Pembelajaran Mesin 2 — Final Project (Individual)
**University:** Universitas Darussalam Gontor

**Topic:** Real-Time Plant Disease Detection Using Lightweight Deep Learning with Transfer Learning

## Experiment Design

| Exp | Model | Description |
|-----|-------|-------------|
| E1 | Custom CNN | Baseline (trained from scratch) |
| E2 | MobileNetV3-Small | Transfer learning (backbone frozen) |
| E3 | MobileNetV3-Small | Transfer learning + data augmentation |
| E4 | MobileNetV3-Small | Transfer learning + fine-tuning |

## How to run

- **Kaggle:** click `+ Add Input` and add the dataset `abdallahalidev/plantvillage-dataset`, then enable **GPU (T4)** in Settings. Run all cells from top to bottom.
- **Google Colab:** run the environment setup cells below (mount Google Drive or upload the dataset zip), then run everything else.

Dataset: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset

## 1. Setup & Imports

In [ ]:
import os
import time
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 2. Data Loading & Preprocessing

### Dataset: PlantVillage (Kaggle — abdallahalidev/plantvillage-dataset)

- 54,306 RGB leaf images, 38 classes (healthy + diseased crops: tomato, potato, apple, grape, corn, ...)
- We use the **color** subset (38 classes). The raw download contains `color/`, `grayscale/`, and `segmented/` subfolders, so the dataset root is auto-detected below.

**Kaggle:** click `+ Add Input` → search `plantvillage-dataset` → add it. Path detection is automatic.
**Colab:** use the setup cells below (mount Google Drive, or upload the zip). Path detection is automatic.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30

WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
RESULTS_DIR = WORK_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Config:')
print(f'  IMG_SIZE   = {IMG_SIZE}')
print(f'  BATCH_SIZE = {BATCH_SIZE}')
print(f'  EPOCHS     = {EPOCHS}')
print(f'  Output dir = {RESULTS_DIR}')

In [ ]:
def find_dataset_dir():
    candidates = []
    if Path('/kaggle/input').exists():
        for ds in Path('/kaggle/input').iterdir():
            if ds.is_dir():
                stack = [ds]
                for _ in range(3):
                    nxt = []
                    for d in stack:
                        candidates.append(d)
                        nxt.extend(c for c in d.iterdir() if c.is_dir())
                    stack = nxt
    if Path('/content').exists():
        candidates += [
            Path('/content/plantvillage-dataset'),
            Path('/content/plantvillage_dataset'),
            Path('/content/plantvillage'),
            Path('/content/drive/MyDrive/plantvillage-dataset'),
            Path('/content/drive/MyDrive/plantvillage_dataset'),
        ]
    best, best_count = None, 0
    for cand in candidates:
        if not cand.exists() or not cand.is_dir():
            continue
        subs = [s for s in cand.iterdir() if s.is_dir()]
        if len(subs) >= 10 and len(subs) > best_count:
            best, best_count = cand, len(subs)
    return best

DATASET_PATH = find_dataset_dir()

if DATASET_PATH is None:
    print('ERROR: dataset not found.')
    print('Kaggle: use "+ Add Input" -> plantvillage-dataset (abdallahalidev).')
    print('Colab : mount Drive with the dataset folder, or upload the zip.')
    raise FileNotFoundError('PlantVillage dataset not found')
else:
    n_class_dirs = len([s for s in DATASET_PATH.iterdir() if s.is_dir()])
    print('Dataset found:', DATASET_PATH)
    print('Number of class folders:', n_class_dirs)

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

all_files = []
all_labels = []
for p in sorted(DATASET_PATH.rglob('*')):
    if p.is_file() and p.suffix.lower() in IMG_EXTS:
        rel = p.relative_to(DATASET_PATH).as_posix()
        all_files.append(rel)
        all_labels.append(str(Path(rel).parent))

df_all = pd.DataFrame({'filename': all_files, 'class': all_labels})
print('Total images:', len(df_all))
print('Number of classes:', df_all['class'].nunique())

train_df, temp_df = train_test_split(df_all, test_size=0.2,
                                     stratify=df_all['class'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5,
                                   stratify=temp_df['class'], random_state=SEED)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
train_datagen = ImageDataGenerator(rescale=1. / 255)
val_datagen = ImageDataGenerator(rescale=1. / 255)
test_datagen = ImageDataGenerator(rescale=1. / 255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, directory=str(DATASET_PATH), x_col='filename', y_col='class',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED)

val_gen = val_datagen.flow_from_dataframe(
    val_df, directory=str(DATASET_PATH), x_col='filename', y_col='class',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False, seed=SEED)

test_gen = test_datagen.flow_from_dataframe(
    test_df, directory=str(DATASET_PATH), x_col='filename', y_col='class',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False, seed=SEED)

class_names = list(train_gen.class_indices.keys())
NUM_CLASSES = len(class_names)
print('Number of classes:', NUM_CLASSES)
print('Class names:', class_names)

In [ ]:
train_counts = pd.Series(train_gen.classes).value_counts().sort_index().values
val_counts = pd.Series(val_gen.classes).value_counts().sort_index().values
test_counts = pd.Series(test_gen.classes).value_counts().sort_index().values

dist_df = pd.DataFrame({'train': train_counts, 'val': val_counts, 'test': test_counts},
                       index=class_names)
print('=== Per-class distribution ===')
print(dist_df.to_string())

ratio = train_counts.max() / train_counts.min()
print(f'\nTrain imbalance ratio (max/min): {ratio:.2f}')
print(f'Total: train={train_counts.sum()}, val={val_counts.sum()}, test={test_counts.sum()}')

## 3. Baseline Model — Custom CNN (E1)

Simple CNN trained from scratch (no pretrained weights). Acts as the baseline for comparison with the transfer-learning models.

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=NUM_CLASSES):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()

In [ ]:
custom_cnn.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

In [ ]:
def make_callbacks(checkpoint_path, patience=6):
    return [
        callbacks.ModelCheckpoint(filepath=checkpoint_path, monitor='val_accuracy',
                                  save_best_only=True, mode='max', verbose=1),
        callbacks.EarlyStopping(monitor='val_loss', patience=patience,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                                    min_lr=1e-6, verbose=1)
    ]

cnn_ckpt = str(RESULTS_DIR / 'custom_cnn_best.h5')

In [ ]:
print('Training Custom CNN (baseline)...')
t0 = time.time()
history_cnn = custom_cnn.fit(train_gen, epochs=EPOCHS, validation_data=val_gen,
                             callbacks=make_callbacks(cnn_ckpt), verbose=1)
print(f'Training time: {(time.time() - t0) / 60:.1f} min')

custom_cnn.load_weights(cnn_ckpt)
val_loss_cnn, val_acc_cnn = custom_cnn.evaluate(val_gen, verbose=0)
print(f'Custom CNN (E1) -> val_acc={val_acc_cnn:.4f}, val_loss={val_loss_cnn:.4f}')

## 4. Main Model — MobileNetV3-Small with Transfer Learning (E2)

Pretrained on ImageNet, backbone frozen. Only the classification head is trained.

In [ ]:
def build_mobilenetv3(num_classes=NUM_CLASSES, trainable=False):
    base = MobileNetV3Small(weights='imagenet', include_top=False,
                            input_shape=(224, 224, 3))
    base.trainable = trainable
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=base.input, outputs=out), base

mnv3, mnv3_base = build_mobilenetv3(trainable=False)
mnv3.summary()

In [ ]:
mnv3.compile(optimizer=optimizers.Adam(learning_rate=1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])
mnv3_ckpt = str(RESULTS_DIR / 'mnv3_transfer_best.h5')

In [ ]:
print('Training MobileNetV3-Small (transfer learning, frozen backbone)...')
t0 = time.time()
history_mnv3 = mnv3.fit(train_gen, epochs=EPOCHS, validation_data=val_gen,
                        callbacks=make_callbacks(mnv3_ckpt), verbose=1)
print(f'Training time: {(time.time() - t0) / 60:.1f} min')

mnv3.load_weights(mnv3_ckpt)
val_loss_mnv3, val_acc_mnv3 = mnv3.evaluate(val_gen, verbose=0)
print(f'MobileNetV3 (E2) -> val_acc={val_acc_mnv3:.4f}, val_loss={val_loss_mnv3:.4f}')

## 5. Experimental Scenarios

### Scenario 2 (E3): MobileNetV3-Small + Data Augmentation

Augmentation is applied to the **training** set only; validation and test sets stay clean (rescaling only), so validation metrics remain valid.

In [ ]:
aug_train_datagen = ImageDataGenerator(
    rescale=1. / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_gen_aug = aug_train_datagen.flow_from_dataframe(
    train_df, directory=str(DATASET_PATH), x_col='filename', y_col='class',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=True, seed=SEED)

mnv3_aug, _ = build_mobilenetv3(trainable=False)
mnv3_aug.compile(optimizer=optimizers.Adam(learning_rate=5e-5),
                 loss='categorical_crossentropy', metrics=['accuracy'])
aug_ckpt = str(RESULTS_DIR / 'mnv3_aug_best.h5')

In [ ]:
print('Training MobileNetV3-Small + augmentation...')
t0 = time.time()
history_aug = mnv3_aug.fit(train_gen_aug, epochs=EPOCHS, validation_data=val_gen,
                           callbacks=make_callbacks(aug_ckpt, patience=8), verbose=1)
print(f'Training time: {(time.time() - t0) / 60:.1f} min')

mnv3_aug.load_weights(aug_ckpt)
val_loss_aug, val_acc_aug = mnv3_aug.evaluate(val_gen, verbose=0)
print(f'MobileNetV3 + Aug (E3) -> val_acc={val_acc_aug:.4f}, val_loss={val_loss_aug:.4f}')

### Scenario 3 (E4): MobileNetV3-Small + Fine-tuning

Two-phase training:
1. Phase 1 — train head only (backbone frozen), lr = 1e-4, 10 epochs
2. Phase 2 — unfreeze the last 20 layers of the backbone, lr = 1e-6, 10 epochs

In [ ]:
mnv3_ft, mnv3_ft_base = build_mobilenetv3(trainable=False)
mnv3_ft.compile(optimizer=optimizers.Adam(learning_rate=1e-4),
                loss='categorical_crossentropy', metrics=['accuracy'])
ft_ckpt = str(RESULTS_DIR / 'mnv3_finetune_best.h5')

In [ ]:
print('Phase 1: train head only (backbone frozen)...')
history_ft_p1 = mnv3_ft.fit(train_gen, epochs=10, validation_data=val_gen,
                            callbacks=make_callbacks(ft_ckpt, patience=8), verbose=1)

print('Phase 2: fine-tune last 20 layers...')
mnv3_ft_base.trainable = True
for layer in mnv3_ft_base.layers[:-20]:
    layer.trainable = False
mnv3_ft.compile(optimizer=optimizers.Adam(learning_rate=1e-6),
                loss='categorical_crossentropy', metrics=['accuracy'])
history_ft_p2 = mnv3_ft.fit(train_gen, epochs=10, validation_data=val_gen,
                            callbacks=make_callbacks(ft_ckpt, patience=8), verbose=1)

history_ft = {k: history_ft_p1.history[k] + history_ft_p2.history[k]
              for k in history_ft_p1.history}

mnv3_ft.load_weights(ft_ckpt)
val_loss_ft, val_acc_ft = mnv3_ft.evaluate(val_gen, verbose=0)
print(f'MobileNetV3 + Fine-tune (E4) -> val_acc={val_acc_ft:.4f}, val_loss={val_loss_ft:.4f}')

## 6. Comparative Analysis (Validation Set)

In [ ]:
results = pd.DataFrame({
    'Experiment': ['E1: Custom CNN (Baseline)', 'E2: MobileNetV3 (Transfer)',
                   'E3: MobileNetV3 + Augmentation', 'E4: MobileNetV3 + Fine-tuning'],
    'Val Accuracy': [val_acc_cnn, val_acc_mnv3, val_acc_aug, val_acc_ft],
    'Val Loss': [val_loss_cnn, val_loss_mnv3, val_loss_aug, val_loss_ft],
})
print(results.to_string(index=False))
results.to_csv(RESULTS_DIR / 'results_val.csv', index=False)

In [ ]:
def plot_history(history, title, save_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].plot(history['accuracy'], label='Train')
    axes[0].plot(history['val_accuracy'], label='Val')
    axes[0].set_title(title + ' - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(history['loss'], label='Train')
    axes[1].plot(history['val_loss'], label='Val')
    axes[1].set_title(title + ' - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / save_name, dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_cnn.history, 'E1: Custom CNN', 'history_cnn.png')
plot_history(history_mnv3.history, 'E2: MobileNetV3 Transfer', 'history_mnv3.png')
plot_history(history_aug.history, 'E3: MobileNetV3 + Aug', 'history_aug.png')
plot_history(history_ft, 'E4: MobileNetV3 + Fine-tune', 'history_ft.png')

## 7. Final Evaluation on Test Set + Error Analysis

The test set (~10%) was never used during training or validation — all numbers below are the final reported metrics.

In [ ]:
def full_eval(model, gen, name):
    gen.reset()
    preds = model.predict(gen, verbose=1)
    y_true = gen.classes
    y_pred = np.argmax(preds, axis=1)
    return {
        'name': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro'),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'y_true': y_true,
        'y_pred': y_pred,
        'cm': confusion_matrix(y_true, y_pred),
    }

models_dict = {
    'E1: Custom CNN (Baseline)': custom_cnn,
    'E2: MobileNetV3 (Transfer)': mnv3,
    'E3: MobileNetV3 + Aug': mnv3_aug,
    'E4: MobileNetV3 + Fine-tune': mnv3_ft,
}

test_results = {}
for name, model in models_dict.items():
    print('\n=== ' + name + ' (test set) ===')
    test_results[name] = full_eval(model, test_gen, name)
    r = test_results[name]
    print(f"acc={r['accuracy']:.4f}  prec={r['precision_macro']:.4f}  "
          f"rec={r['recall_macro']:.4f}  f1={r['f1_macro']:.4f}")

test_summary = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('y_true', 'y_pred', 'cm')}
    for r in test_results.values()
])
print('\n=== Test Set Summary ===')
print(test_summary.to_string(index=False))
test_summary.to_csv(RESULTS_DIR / 'results_test.csv', index=False)

In [ ]:
best_name = max(test_results, key=lambda k: test_results[k]['accuracy'])
best_res = test_results[best_name]
print('Best model on test set:', best_name, '(acc=%.4f)' % best_res['accuracy'])

print('\nClassification report (best model):')
print(classification_report(best_res['y_true'], best_res['y_pred'],
                            target_names=class_names, digits=4, zero_division=0))

plt.figure(figsize=(22, 18))
sns.heatmap(best_res['cm'], annot=False, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - ' + best_name)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mis_idx = np.where(best_res['y_true'] != best_res['y_pred'])[0]
print('Misclassified:', len(mis_idx), '/', len(best_res['y_true']))

sample_idx = mis_idx[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, i in zip(axes.ravel(), sample_idx):
    rel_path = test_df.iloc[i]['filename']
    img = plt.imread(DATASET_PATH / rel_path)
    ax.imshow(img)
    ax.set_title('True: ' + class_names[best_res['y_true'][i]].split('___')[-1] +
                 '\nPred: ' + class_names[best_res['y_pred'][i]].split('___')[-1],
                 fontsize=9)
    ax.axis('off')
plt.suptitle('Misclassified Samples - ' + best_name, fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'misclassified_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def measure_inference_time(model, n=100):
    test_gen.reset()
    xb, _ = next(test_gen)
    model.predict(xb[:1], verbose=0)
    times = []
    for _ in range(n):
        test_gen.reset()
        xb, _ = next(test_gen)
        t0 = time.time()
        model.predict(xb[:1], verbose=0)
        times.append((time.time() - t0) * 1000)
    return float(np.mean(times))

for name, model in models_dict.items():
    t_ms = measure_inference_time(model)
    print(name + ': %.2f ms/image (GPU)' % t_ms)

In [ ]:
val_acc_map = {
    'E1: Custom CNN (Baseline)': val_acc_cnn,
    'E2: MobileNetV3 (Transfer)': val_acc_mnv3,
    'E3: MobileNetV3 + Aug': val_acc_aug,
    'E4: MobileNetV3 + Fine-tune': val_acc_ft,
}

summary_json = {
    'seed': SEED,
    'tensorflow_version': tf.__version__,
    'gpu': [d.name for d in tf.config.list_physical_devices('GPU')],
    'dataset': {
        'path': str(DATASET_PATH),
        'total_images': int(len(df_all)),
        'num_classes': NUM_CLASSES,
        'train': int(len(train_df)),
        'val': int(len(val_df)),
        'test': int(len(test_df)),
        'train_imbalance_ratio': float(ratio),
        'class_names': class_names,
    },
    'config': {'img_size': list(IMG_SIZE), 'batch_size': BATCH_SIZE, 'epochs': EPOCHS},
    'models': {},
}

for name, r in test_results.items():
    summary_json['models'][name] = {
        'val_accuracy': float(val_acc_map[name]),
        'test_accuracy': float(r['accuracy']),
        'precision_macro': float(r['precision_macro']),
        'recall_macro': float(r['recall_macro']),
        'f1_macro': float(r['f1_macro']),
    }

histories = {
    'E1: Custom CNN (Baseline)': history_cnn.history,
    'E2: MobileNetV3 (Transfer)': history_mnv3.history,
    'E3: MobileNetV3 + Aug': history_aug.history,
    'E4: MobileNetV3 + Fine-tune': history_ft,
}
summary_json['histories'] = {k: {m: list(v[m]) for m in v} for k, v in histories.items()}
summary_json['best_model'] = best_name
summary_json['confusion_matrix_best'] = best_res['cm'].tolist()

with open(RESULTS_DIR / 'experiment_results.json', 'w') as f:
    json.dump(summary_json, f, indent=2)
print('Saved:', RESULTS_DIR / 'experiment_results.json')
print('All artifacts in:', RESULTS_DIR)
print(sorted(p.name for p in RESULTS_DIR.iterdir()))

## 8. Conclusion & Reproducibility

- All results (validation + test metrics, confusion matrix, training histories) are saved in `/kaggle/working/results/` (Kaggle) or `/content/results/` (Colab).
- Download `experiment_results.json` — it contains every number needed for the final report.
- Code: https://github.com/FarrelGhozy/TI5A2_452024611053_PembelanganMesin2_FarrelGhozy

In [ ]:
print('=== FINAL SUMMARY (TEST SET) ===')
for name, r in test_results.items():
    print(name + ': acc=%.4f | prec=%.4f | rec=%.4f | f1=%.4f'
          % (r['accuracy'], r['precision_macro'], r['recall_macro'], r['f1_macro']))
print('\nBest:', best_name)

## End of Notebook